In [1]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

# 1. Read the books and remove blank lines
books = []

for file_path in sorted(Path("books_text").glob("*.md")):
    text = file_path.read_text(encoding="utf-8")

    books.append({
        "source": file_path.name,
        "content": [
            line
            for line in text.splitlines()
            if line.strip()
        ],
    })

if not books:
    raise FileNotFoundError(
        "No Markdown files found in books_text/. "
        "Check your notebook's working directory."
    )

# 2. Create chunks
chunks = chunk_documents(books, size=100, step=50)

# 3. Define the function before calling it
def prepare_documents(chunks):
    documents = []

    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)

    return documents

# 4. Prepare and index the chunks
documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print("Number of documents indexed:", len(documents))

results = index.search(
    "python function definition",
    num_results=5,
)

if results:
    print("Top result came from:", results[0]["source"])
    print("\nContent preview:")
    print(results[0]["content"][:1000])
else:
    print("No results found.")


Number of documents indexed: 988
Top result came from: PhysicalModelingInMatlab4.md

Content preview:
| ------ | ------------ | --- | --- |
| total  | = total +    | a;  |     |
end
ans = total
Ifyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised
to find, after running the script, that their values had changed. If you have two scripts that
48 Functions
use the same variable names, you might find that they work separately and then break when
you try to combine them. This kind of interaction is called a collision.
name
As the number of scripts you write increases, and they get longer and more complex, name
collisions become more of a problem. Avoiding this problem is one of several motivations for
functions.
| 5.2 | Defining | Functions |     |     |
| --- | -------- | --------- | --- | --- |
A function is like a script, except that each function has its own workspace, so any variables
defined inside a function only exist while the function is running and don

In [2]:
from openai import OpenAI

openai_client = OpenAI()
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context,
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages,
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [3]:
result = rag("python function definition")

print("Response:\n")
print(result["answer"])

print("\nInput tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])

Response:

To define a Python function, follow this general syntax:

```python
def function_name(parameters):
    # Function body: code that performs a task
    return result  # Optional: returns a value
```

### Explanation:
1. **Keyword `def`**: This indicates that you are defining a function.
2. **Function Name**: Replace `function_name` with the desired name for your function. It should ideally be descriptive of what the function does.
3. **Parameters**: You can specify one or more parameters (input variables) inside the parentheses. Use commas to separate them.
4. **Function Body**: Indent the code block to define the functionality of the function. This is where you write the steps the function will execute.
5. **Return Statement**: If you want the function to return a value, use the `return` keyword followed by the value you want to return. If no return statement is included, the function will return `None` by default.

### Example:
Here's a simple function that adds two numbers: